In [ ]:
## Nikolay Vorontsov,
## Mushroom task
## Inference with fine-tuned model

In [ ]:
!pip install transformers huggingface_hub pip torch jsonlines regex
# Install necessary dependencies
!pip install transformers peft accelerate huggingface_hub
!pip install -q trl xformers wandb datasets einops sentencepiece
!pip install -U datasets bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.4/293.4 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 103.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 17.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 MB 10.3 MB/s eta 0:00:00


In [ ]:
import jsonlines
import re
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

from huggingface_hub import login

# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-generation", model="meta-llama/Llama-2-7b-hf")

from google.colab import userdata

HUGGING_API = userdata.get('HUGGINGFACE_READ_AND_WRITE')

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-2-7b-hf.
401 Client Error. (Request ID: Root=1-677e1e41-13991c5718ad99143752921c;9dc51746-73d3-4f59-9681-e2790fbab024)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-2-7b-hf/resolve/main/config.json.
Access to model meta-llama/Llama-2-7b-hf is restricted. You must have access to it and be authenticated to access it. Please log in.

In [ ]:

# Login to Hugging Face
login(token=HUGGING_API)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("nicksnlp/llama-7B-hallucination")
model = AutoModelForTokenClassification.from_pretrained("nicksnlp/llama-7B-hallucination")

# Check for CUDA availability and set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
"""
def infer_with_model(input_text):
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
    predicted_labels = torch.argmax(logits, dim=-1)
    tokens = tokenizer.tokenize(input_text)
    labeled_tokens = list(zip(tokens, predicted_labels[0].tolist()))
    hallucinated_words = [token for token, label in labeled_tokens if label == 1]
    return hallucinated_words
"""
def infer_with_model(input_text):
    # Tokenize the input text
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=128)

    # Move input tensors to the same device as the model
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # Predict the token labels (hallucination vs. correct)
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits  # Raw logits output from the model

    # Get the predicted labels (0 for correct, 1 for hallucinated)
    predicted_labels = torch.argmax(logits, dim=-1)

    # Decode the tokens from the input text
    tokens = tokenizer.tokenize(input_text)

    # Get the corresponding predicted labels for each token
    labeled_tokens = list(zip(tokens, predicted_labels[0].tolist()))

    # Create a list of hallucinated words
    hallucinated_words = [token for token, label in labeled_tokens if label == 1]

    return hallucinated_words

def find_spans(model_output_text, hallucinated_words):
    spans = []
    for word in hallucinated_words:
        for match in re.finditer(re.escape(word), model_output_text):
            spans.append([match.start(), match.end()])
    return spans

def group_adjacent_words(hallucinated_words): # "New" "New York" -> "New York"
    grouped_words = []
    i = 0
    while i < len(hallucinated_words):
        current_group = [hallucinated_words[i]]
        j = i + 1
        while j < len(hallucinated_words):
            if j < len(hallucinated_words) and hallucinated_words[j].startswith(hallucinated_words[i] + " "):
                current_group.append(hallucinated_words[j])
                i = j
            elif j < len(hallucinated_words) and hallucinated_words[i].startswith(hallucinated_words[j] + " "):
                current_group.insert(0, hallucinated_words[j])
                i = j
            else:
                break
            j += 1
        grouped_words.append(current_group[-1])
        i += 1
    return grouped_words

def process_validation_set(validation_file, output_file):
    with jsonlines.open(validation_file) as reader, jsonlines.open(output_file, 'w') as writer:
        for datapoint in reader:
            model_output_text = datapoint.get("model_output_text", "")
            input_text = datapoint.get("input_text", "")

            hallucinated_words = infer_with_model(input_text)

            decoded_hallucinated_words = [tokenizer.decode(tokenizer.convert_tokens_to_ids(word)).strip() for word in hallucinated_words]
            decoded_hallucinated_words = list(filter(None, decoded_hallucinated_words))

            grouped_hallucinated_words = group_adjacent_words(decoded_hallucinated_words)

            spans = find_spans(model_output_text, grouped_hallucinated_words)
            datapoint["hallucinated_spans"] = spans
            writer.write(datapoint)


tokenizer_config.json:   0%|          | 0.00/978 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.62M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/437 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.22k [00:00<?, ?B/s]

Unused kwargs: ['_load_in_4bit', '_load_in_8bit', 'quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.
`low_cpu_mem_usage` was None, now default to True since model is quantized.


model.safetensors:   0%|          | 0.00/3.91G [00:00<?, ?B/s]

In [ ]:

# Example usage:
validation_file = "/content/mushroom.en-val.v2.unlabeled.jsonl"
output_file = "validation_with_spans.jsonl"
process_validation_set(validation_file, output_file)
print(f"Processed data written to {output_file}")

Processed data written to validation_with_spans.jsonl
